# Ejercicio 11: Web Scraping

## Objetivo de la práctica

El objetivo de este ejercicio es construir un web scraper que recoja datos de un website.

### Parte 0: Planificar
1. Identificar los datos que quieres obtener.
2. Elegir el sitio web objetivo.
3. Planificar la estructura del corpus.

## Parte 1: Entender el sitio web objetivo

- Analizar la estructura de la página web a ser analizada.
- Identificar los elementos HTML que contienen los datos bsuscados.

In [8]:
from bs4 import BeautifulSoup

file = '/kaggle/input/datasets/dannyconstante/datahtml/rotisserie-chicken.html'

# Load the HTML file
with open(file, "r", encoding="utf-8") as file:
    html_content = file.read()
    
# Parse the HTML content with BeautifulSoup
soup = BeautifulSoup(html_content, "html.parser")

In [10]:
# Extracting the recipe title
title = soup.find("meta", {"property": "og:title"})["content"]
title

'Rotisserie Chicken'

In [11]:
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
for ingredient in ingredients_section:
    print(ingredient.text.strip())

1 (3 pound) whole chicken
1 pinch salt
¼ cup butter, melted
1 tablespoon salt
1 tablespoon ground paprika
¼ tablespoon ground black pepper


## Parte 2: Obtener los datos deseados

* Buscar dentro del contenido HTML y extraer la información.

In [12]:
# Extracting the description
description = soup.find("meta", {"name": "description"})["content"]

# Extracting the ingredients
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
ingredients = [ingredient.get_text().strip() for ingredient in ingredients_section]

# Extracting the instructions
instructions_section = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
instructions = [instruction.get_text().strip() for instruction in instructions_section]

# Extracting the nutrition information
nutrition_section = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
nutrition_facts = [fact.parent.get_text().strip().replace('\n', ' ') for fact in nutrition_section]

# Print the extracted information
print("Recipe Title:", title)
print("Description:", description)
print("Ingredients:")
for ingredient in ingredients:
    print("-", ingredient)
print("Instructions:")
for i, instruction in enumerate(instructions, 1):
    print(f"{i}. {instruction}")
print("Nutrition Facts:")
for fact in nutrition_facts:
    print("-", fact)


Recipe Title: Rotisserie Chicken
Description: Rotisserie chicken that's easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.
Ingredients:
- 1 (3 pound) whole chicken
- 1 pinch salt
- ¼ cup butter, melted
- 1 tablespoon salt
- 1 tablespoon ground paprika
- ¼ tablespoon ground black pepper
Instructions:
1. Intimidated by the idea of making a rotisserie chicken at home? We're here to help. Get your grill and rotisserie attachment ready — you'll want to try this recipe ASAP.
2. Here's what you'll need to make rotisserie chicken at home:
3. · Whole Chicken: This recipe is meant for a whole 3-pound chicken. If your chicken is larger or smaller, you'll have to adjust the cooking time.· Butter: Butter keeps the chicken moist and juicy, while giving the seasonings something to stick to.· Seasonings: The rotisserie chicken is simply seasoned with salt, pepper, and paprika.
4. You'll find the full, step-by-step recipe below — b

## Parte 3: Obtener enlaces relacionados
* Encontrar links a otras recetas para completar el corpus

In [13]:
# Find all the links to other recipes
recipe_links = soup.find_all("a", href=True)

# Filter and print only the links that are likely to be recipes
recipe_urls = []
for link in recipe_links:
    href = link['href']
    if "recipe" in href:
        recipe_urls.append(href)

# Print the recipe URLs
print("Linked Recipes:")
for url in recipe_urls:
    print(url)

Linked Recipes:
https://www.allrecipes.com/authentication/login?regSource=3675&relativeRedirectUrl=%2Frecipe%2F93168%2Frotisserie-chicken%2F
/account/add-recipe
https://www.myrecipes.com/favorites
https://www.allrecipes.com/authentication/logout?relativeRedirectUrl=%2Frecipe%2F93168%2Frotisserie-chicken%2F
https://www.magazines.com/allrecipes-magazine.html?utm_source=allrecipes.com&utm_medium=owned&utm_campaign=i111arr1w2661
https://www.magazines.com/allrecipes-magazine.html
https://www.allrecipes.com/recipes/17562/dinner/
https://www.allrecipes.com/recipes/17057/everyday-cooking/more-meal-ideas/5-ingredients/main-dishes/
https://www.allrecipes.com/recipes/15436/everyday-cooking/one-pot-meals/
https://www.allrecipes.com/recipes/1947/everyday-cooking/quick-and-easy/
https://www.allrecipes.com/recipes/455/everyday-cooking/more-meal-ideas/30-minute-meals/
https://www.allrecipes.com/recipes/17889/everyday-cooking/family-friendly/family-dinners/
https://www.allrecipes.com/recipes/94/soups-s

### Construcción del Corpus mediante Web Crawling
En esta sección, automatizamos la recolección de datos para construir nuestro corpus de conocimiento estructurado. El pipeline realiza los siguientes pasos:

1. **Extracción y Filtrado de Enlaces:** A partir del archivo procesado, extraemos todos los hipervínculos (`<a href>`). Se aplica un filtro estricto para conservar únicamente aquellos enlaces que apuntan a recetas válidas (verificando el prefijo `http` y el path `/recipe/`).
2. **Web Crawling:** Un script iterativo visita automáticamente cada uno de los enlaces filtrados, simulando la petición de un navegador web.
3. **Extracción y Transformación (Scraping):** En cada página, se extrae el título, los ingredientes y las instrucciones. Estos datos se consolidan en un único bloque de texto (`texto_completo_rag`), optimizado para preservar el contexto semántico.
4. **Almacenamiento (Corpus):** La información procesada se almacena en un DataFrame de `pandas` y se exporta en formato CSV. Este archivo conformará la base de conocimiento (documentos) que consumirá nuestro sistema RAG en la siguiente fase.

In [20]:
import os
import pandas as pd
from bs4 import BeautifulSoup

# ---------------------------------------------------------
# PASO 1: DEFINIR LA RUTA LOCAL DE LOS ARCHIVOS HTML
# ---------------------------------------------------------
# Ruta exacta de tu dataset en Kaggle
directorio_html = "/kaggle/input/datasets/dannyconstante/recipes-html"

# Verificamos que la ruta exista antes de continuar
if not os.path.exists(directorio_html):
    print(f"❌ ERROR: No se encontró la ruta {directorio_html}")
else:
    print(f"📂 Accediendo al directorio: {directorio_html}")

lista_corpus = []
archivos_procesados = 0

# ---------------------------------------------------------
# PASO 2: ITERAR Y EXTRAER DATOS DE CADA ARCHIVO LOCAL
# ---------------------------------------------------------
# Recorremos todos los archivos dentro de la carpeta
for nombre_archivo in os.listdir(directorio_html):
    
    # Nos aseguramos de procesar únicamente los archivos web
    if nombre_archivo.endswith(".html") or nombre_archivo.endswith(".htm"):
        ruta_completa = os.path.join(directorio_html, nombre_archivo)
        
        try:
            # Abrimos el archivo localmente con codificación utf-8
            with open(ruta_completa, 'r', encoding='utf-8') as archivo_local:
                contenido = archivo_local.read()
                
            s = BeautifulSoup(contenido, 'html.parser')
            
            # 1. Extraer Título
            title_elem = s.find('h1')
            title = title_elem.text.strip() if title_elem else nombre_archivo.replace('.html', '')
            
            # 2. Extraer Ingredientes
            ing_section = s.find('div', id='mntl-structured-ingredients_1-0')
            if ing_section:
                ingredients = [item.text.strip() for item in ing_section.find_all(['p', 'li'])]
            else:
                ingredients = ["Ingredientes no encontrados"]
                
            # 3. Extraer Instrucciones
            inst_section = s.find('div', id='recipe__steps_1-0')
            if inst_section:
                instructions = [item.text.strip() for item in inst_section.find_all(['p', 'li'])]
            else:
                instructions = ["Instrucciones no encontradas"]
            
            # 4. Formatear el texto unificado para el sistema RAG
            ingredientes_texto = "\n".join(ingredients)
            instrucciones_texto = "\n".join(instructions)
            
            texto_rag = f"Título: {title}\n\nIngredientes:\n{ingredientes_texto}\n\nInstrucciones:\n{instrucciones_texto}"
            
            # 5. Guardar en nuestra lista
            lista_corpus.append({
                "archivo_origen": nombre_archivo,
                "titulo": title,
                "texto_completo_rag": texto_rag
            })
            
            archivos_procesados += 1
            # Imprimimos de 5 en 5 para no saturar la salida de la celda en Kaggle
            if archivos_procesados % 5 == 0:
                print(f"✔️ Procesados {archivos_procesados} archivos...")
                
        except Exception as e:
            print(f"❌ Error al leer el archivo {nombre_archivo}: {e}")

# ---------------------------------------------------------
# PASO 3: CREAR EL DATA FRAME Y EXPORTAR (EL CORPUS REAL)
# ---------------------------------------------------------
if len(lista_corpus) > 0:
    df_corpus = pd.DataFrame(lista_corpus)
    print(f"\n🚀 ¡Extracción finalizada! Se procesaron {len(lista_corpus)} recetas en total.")
    
    # Mostrar las primeras filas en el cuaderno para validar
    display(df_corpus.head())

    # Guardar el corpus definitivo en el directorio de salida (working) de Kaggle
    ruta_salida = "/kaggle/working/corpus_recetas_allrecipes.csv"
    df_corpus.to_csv(ruta_salida, index=False)
    print(f"✅ Archivo '{ruta_salida}' generado con éxito. Listo para la Parte 4.")
else:
    print("\n❌ La lista final está vacía, revisa si los archivos HTML están en la ruta correcta.")

📂 Accediendo al directorio: /kaggle/input/datasets/dannyconstante/recipes-html
✔️ Procesados 5 archivos...
✔️ Procesados 10 archivos...

🚀 ¡Extracción finalizada! Se procesaron 14 recetas en total.


,archivo_origen,titulo,texto_completo_rag
0,Instant Pot Crispy Barbecue Chicken Wings Reci...,Instant Pot Crispy Barbecue Chicken Wings,Título: Instant Pot Crispy Barbecue Chicken Wi...
1,Zesty Slow Cooker Chicken Barbecue Recipe.html,Zesty Slow Cooker Chicken Barbecue,Título: Zesty Slow Cooker Chicken Barbecue\n\n...
2,Carolina-Style _Whole Hog_ Barbecue Pork Recip...,"Carolina-Style ""Whole Hog"" Barbecue Pork","Título: Carolina-Style ""Whole Hog"" Barbecue Po..."
3,Oven-Roasted Ribs Recipe.html,Oven-Roasted Ribs,Título: Oven-Roasted Ribs\n\nIngredientes:\nIn...
4,Slow-Cooker Barbecue Ribs Recipe.html,Slow-Cooker Barbecue Ribs,Título: Slow-Cooker Barbecue Ribs\n\nIngredien...


✅ Archivo '/kaggle/working/corpus_recetas_allrecipes.csv' generado con éxito. Listo para la Parte 4.


## Parte 4: Hacer RAG con las recetas obtenidas
* Una vez que se ha construido el corpus, implementar y desplegar RAG para realizar búsquedas en el corpus

In [21]:
# 0. Instalar ChromaDB de forma silenciosa en Kaggle
!pip install chromadb -q

import pandas as pd
import chromadb

print("Iniciando el sistema RAG...")

# 1. Cargar el corpus desde el directorio de trabajo de Kaggle
ruta_csv = "/kaggle/working/corpus_recetas_allrecipes.csv"
df = pd.read_csv(ruta_csv)

# 2. Inicializar el cliente de ChromaDB
chroma_client = chromadb.Client()

# Limpiamos la colección si la celda se vuelve a ejecutar para evitar duplicados
nombre_coleccion = "coleccion_recetas_bbq"
try:
    chroma_client.delete_collection(name=nombre_coleccion)
except:
    pass

coleccion = chroma_client.create_collection(name=nombre_coleccion)

# 3. Preparar los datos para la vectorización
documentos = df['texto_completo_rag'].tolist()
# Ahora guardamos el nombre del archivo de origen como metadato
metadatos = [{"titulo": row['titulo'], "archivo": str(row['archivo_origen'])} for _, row in df.iterrows()]
ids = [str(i) for i in range(len(df))]

print(f"Vectorizando {len(documentos)} recetas e insertándolas en la base de datos...")

# 4. Ingesta de datos (Aquí ocurre la magia de los embeddings)
coleccion.add(
    documents=documentos,
    metadatas=metadatos,
    ids=ids
)

print("✅ ¡Base de datos vectorial lista para ser consultada!\n")
print("-" * 60)

# ---------------------------------------------------------
# FASE DE RECUPERACIÓN (RETRIEVAL)
# ---------------------------------------------------------

def buscar_recetas_rag(query, n_resultados=2):
    print(f"🔍 Buscando recetas para la consulta: '{query}'\n")
    
    resultados = coleccion.query(
        query_texts=[query],
        n_results=n_resultados
    )
    
    for i in range(len(resultados['documents'][0])):
        doc = resultados['documents'][0][i]
        meta = resultados['metadatas'][0][i]
        distancia = resultados['distances'][0][i] # Menor distancia = Mayor similitud
        
        print(f"🍽️ TÍTULO: {meta['titulo']}")
        print(f"📁 ARCHIVO: {meta['archivo']}")
        print(f"📊 Similitud (Distancia L2): {distancia:.4f}")
        print("📝 FRAGMENTO RECUPERADO (Contexto para inyectar en un LLM):")
        # Mostramos los primeros 350 caracteres para validar la información
        print(doc[:350] + "...\n")
        print("-" * 60)

# 5. Prueba de tu motor de búsqueda
# NOTA: Las búsquedas deben ser en inglés porque los textos extraídos de los HTML están en inglés.
mi_pregunta = "I want a spicy chicken recipe with pepper and garlic"
buscar_recetas_rag(mi_pregunta, n_resultados=2)

Iniciando el sistema RAG...
Vectorizando 14 recetas e insertándolas en la base de datos...


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [01:45<00:00, 789kiB/s] 


✅ ¡Base de datos vectorial lista para ser consultada!

------------------------------------------------------------
🔍 Buscando recetas para la consulta: 'I want a spicy chicken recipe with pepper and garlic'

🍽️ TÍTULO: Instant Pot Crispy Barbecue Chicken Wings
📁 ARCHIVO: Instant Pot Crispy Barbecue Chicken Wings Recipe.html
📊 Similitud (Distancia L2): 0.9911
📝 FRAGMENTO RECUPERADO (Contexto para inyectar en un LLM):
Título: Instant Pot Crispy Barbecue Chicken Wings

Ingredientes:
Ingredientes no encontrados

Instrucciones:
Instrucciones no encontradas...

------------------------------------------------------------
🍽️ TÍTULO: Zesty Slow Cooker Chicken Barbecue
📁 ARCHIVO: Zesty Slow Cooker Chicken Barbecue Recipe.html
📊 Similitud (Distancia L2): 0.9968
📝 FRAGMENTO RECUPERADO (Contexto para inyectar en un LLM):
Título: Zesty Slow Cooker Chicken Barbecue

Ingredientes:
Ingredientes no encontrados

Instrucciones:
Instrucciones no encontradas...

-------------------------------------------